In [ ]:
using Bessels
include("/Users/travail/Desktop/Quantic/Dev/Stage_quantic_2025/Code_Julia/Julia_file/classical_oscillators.jl");

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####

ω_a = 4*2*pi
ω_b_0 = 7.05*2*pi

Lamb_shift_b = - 0.05 #Lamb shift in GHz for the mode b
Lamb_shift_a = - 0.0 #Lamb shift in GHz for the mode a

ω_b = ω_b_0 #- Lamb_shift*2*pi # Shifted frequency of mode b

φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.2
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b
E_Cb = C_b_inv/2

E_J = 37*2*pi




κ_b=1/10; # 1/ns from Leon's paper
ϕ_quantum=1/2

ω_d=ω_b_0 + Lamb_shift_b*2*pi# Drive frequency
ω_p=2ω_a-ω_d # Pump Frequency

ϵ_0 = 0.6*pi
g2_max = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=0.75*g2_max

κ_2 = 4*g2_max^2/κ
κ_a = 3e-3*κ_2; #from Leon's paper

In [ ]:
κ_a

In [ ]:
κ_b*C_b_inv

In [ ]:
n_huge=3000
AB, J = Exp_decay(bath_params=(ω_c=50.0, ω_cut_off=20*2*pi),κ=κ_b, C_inv=C_b_inv)
params = chaincoeffs_finiteT(n_huge, β, false; J=J, mc=size(AB)[1], mp=0, AB=AB, iq=1, idelta=2, procedure=:Lanczos, Mmax=5000, save=false)  # chain parameters, i.e. on-site energies ϵ_i, hopping energies t_i, and system-chain coupling c_0
t_max = 8/κ
n_opt = findchainlength_dich(t_max, params; eps=10^-4, verbose=true)
n=n_opt+10

In [ ]:
n_a=2000
n_b=2000

t_max=8/κ

bath=Exp_decay
bath_params=(ω_c=50*2*pi,ω_cut_off=20*2*pi);

In [ ]:
# Initialization of the bath parameters
AB_a, J_a = bath(bath_params=bath_params,κ=κ_a, C_inv=C_a_inv)
chain_params_a= chaincoeffs_finiteT(n_a, β, false; J=J_a, mc=size(AB_a)[1], mp=0, AB=AB_a, iq=1, idelta=2, procedure=:Lanczos, Mmax=5000, save=false)  # chain parameters, i.e. on-site energies ϵ_i, hopping energies t_i, and system-chain coupling c_0

AB_b, J_b = bath(bath_params=bath_params,κ=κ_b, C_inv=C_b_inv)
chain_params_b = chaincoeffs_finiteT(n_b, β, false; J=J_b, mc=size(AB_b)[1], mp=0, AB=AB_b, iq=1, idelta=2, procedure=:Lanczos, Mmax=5000, save=false)  # chain parameters, i.e. on-site energies ϵ_i, hopping energies t_i, and system-chain coupling c_0

In [ ]:
ϕ_a,ϕ_b,_,q_a,q_b,_,t_list = two_plus_one_photon_chain(n_a, n_b, t_max; chain_params_a=chain_params_a,chain_params_b=chain_params_b,
     circuit_params=(ω_a=ω_a,ω_b=ω_b,E_J = E_J, E_Ca=E_Ca, E_Cb=E_Cb, ϵ_0 = ϵ_0,κ_a= κ_a,κ_b=κ_b,ϵ_d=ϵ_d,ω_d=ω_d),
     init_mode_a=(ϕ_a_0=0,q_a_0 = 0), init_mode_b=(ϕ_b_0=0,q_b_0 = 0), β=β,minimal_a=false)

In [ ]:
function drive(t)
    return 0*ϵ_d * cos(ω_d * t) #We consider a drive resonant with the mode b
end

ω_p=2ω_a-ω_d
function epsilon(t)
    return 0*ϵ_0 * cos(ω_p * t) # We consider a drive resonant with the two-photon process
end


temp_buf_a = zeros(n_a)
temp_buf_b = zeros(n_b)
function two_photon_ODE(ddu,du,u,p,t)
    ϕ_a = u[1]
    ϕ_b = u[2]
    Φ_a=u[3:n_a+2]
    Φ_b=u[n_a+3:end]

    ddu[1] = -ω_a^2*ϕ_a + 2*E_J*ω_a * sqrt(Z_a) * sin(epsilon(t))*(cos(ϕ_a * sqrt(Z_a)/ϕ_quantum + ϕ_b * sqrt(Z_b)/ϕ_quantum) - 1) / ϕ_quantum + ω_a * system_chain_coupling_a * sqrt(Z_a) * Φ_a[1]
    ddu[2] = -ω_b^2*ϕ_b + ω_b * system_chain_coupling_b * sqrt(Z_b) * Φ_b[1] - ω_b *drive(t)*sqrt(2) + 2*E_J*ω_b * sqrt(Z_b) * sin(epsilon(t))*(cos(ϕ_a * sqrt(Z_a)/ϕ_quantum + ϕ_b * sqrt(Z_b)/ϕ_quantum) - 1) / ϕ_quantum
    mul!(temp_buf_a, Mϕ_chain_a, Φ_a)
    @. temp_buf_a = -temp_buf_a
    temp_buf_a[1] += system_chain_coupling_a * sqrt(Z_a) * ϕ_a
    mul!(view(ddu, 3:n_a+2), Mq_chain_a, temp_buf_a)

    mul!(temp_buf_b, Mϕ_chain_b, Φ_b)
    @. temp_buf_b = -temp_buf_b
    temp_buf_b[1] += system_chain_coupling_b * sqrt(Z_b) * ϕ_b
    mul!(view(ddu, n_a+3:length(ddu)), Mq_chain_b, temp_buf_b)
    return nothing
end


In [ ]:
gq_a=vcat(- hopping_energies_a)
gϕ_a=vcat(hopping_energies_a)

Mq_chain_a=SymTridiagonal(onsite_energies_a, gq_a)
Mϕ_chain_a=SymTridiagonal(onsite_energies_a, gϕ_a)

gq_b=vcat(- hopping_energies_b)
gϕ_b=vcat(hopping_energies_b)
Mq_chain_b=SymTridiagonal(onsite_energies_b, gq_b)
Mϕ_chain_b=SymTridiagonal(onsite_energies_b, gϕ_b)

Mq_full=SymTridiagonal(vcat([ω_a,ω_b], onsite_energies_a, onsite_energies_b), vcat([0,0],gq_a,[0] ,gq_b))

#Initial conditions
ϕ_a_0,q_a_0 = (0,0)

ϕ_b_0,q_b_0 = (0,0)


ϕ_0 = vcat(ϕ_a_0/sqrt(Z_a),ϕ_b_0/sqrt(Z_b), fill(0.0, n_a+n_b))
q_0 = vcat(q_a_0 * sqrt(Z_a),q_b_0 * sqrt(Z_b), fill(0.0, n_a+n_b))
ϕ_dot_0 = Mq_full*q_0


#Run the simulation
tspan = (0.0, t_max) # Time span for the simulation
dt = 1 / (5 * maximum(vcat(ω_p,ω_a,ω_b,onsite_energies_a,onsite_energies_b))) # Time step for the simulation
prob = SecondOrderODEProblem(two_photon_ODE, ϕ_dot_0, ϕ_0, tspan)
sol = solve(prob, VelocityVerlet(),dt=dt)

ϕ_dot = [getindex.(sol.u,i) for i in 1:n_a+n_b+2]
ϕ = [getindex.(sol.u,i+n_a+n_b+2) for i in 1:n_a+n_b+2]

ϕ_a = ϕ[1] * sqrt(Z_a)
ϕ_b = ϕ[2] * sqrt(Z_b)

fac = lu(Mq_full)
q = [fac \ getindex.(ϕ_dot,t) for t in 1:length(sol.t)]
q = [getindex.(q,i) for i in 1:n_a+n_b+2]
q_a = q[1] / sqrt(Z_a)
q_b = q[2] / sqrt(Z_b)

t_list = sol.t;

In [ ]:
plot(t_list*κ_b, ϕ_b)

In [ ]:
F,freqs=FFT_perso(ϕ_b, t_list)
f_shifted=abs(freqs[argmax(abs.(F))])
plot(freqs,abs.(F))
vline!([f_shifted], linestyle=:dash, color=:red, label="f_shifted")
plot!(xlims=(3,8))

In [ ]:
f_shifted - ω_b/(2*pi)

In [ ]:
Re_a = ϕ_a / sqrt(2*Z_a)
Im_a = q_a * sqrt(Z_a/2)

ϕ_rotating_a =@. ϕ_a * cos(ω_a * t_list) + q_a*Z_a* sin(-ω_a * t_list)
q_rotating_a =@. -ϕ_a/Z_a * sin(-ω_a * t_list) + q_a* cos(ω_a * t_list)

Re_rotating_a = ϕ_rotating_a / sqrt(2*Z_a)
Im_rotating_a = q_rotating_a * sqrt(Z_a/2)

# ϕ_rotating_b =@. ϕ_CL_b * cos(ω_a * t_list) + q_CL_b*Z_a* sin(-ω_a * t_list)
# q_rotating_b =@. -ϕ_CL_b/Z_a * sin(-ω_a * t_list) + q_CL_b* cos(ω_a * t_list);
p1=plot(Re_a,Im_a)
p2=plot(Re_rotating_a,Im_rotating_a)

layout = (1,2)
plot(p1,p2,layout=layout, xlabel=L"\Re(a)", ylabel=L"\Im(a)", title=["Resonator a" "Resonator a RF"], legend=false,
size=(layout[2], layout[1]).*size_px_car,
margin = 10Plots.mm)
#savefig("cat_state_chain.pdf")

In [ ]:
ω_a_CL = ω_a
ω_b_CL = ω_b_0+0.05*2*pi
ϕ_CL_a, ϕ_CL_b, q_CL_a, q_CL_b, t_list_CL = two_photon_CL(10/κ_b;
        circuit_params=(ω_a=ω_a_CL,ω_b=ω_b_CL, E_J = E_J, E_Ca=E_Ca, E_Cb=E_Cb, ϵ_0 = ϵ_0,κ_a=κ_a,κ_b=κ_b,ϵ_d=ϵ_d,ω_d=ω_b_CL),
        init_mode_a=(ϕ_a_0=0.0,q_a_0 = 0.0), init_mode_b=(ϕ_b_0=0.0,q_b_0 = 0.0), β=∞)

In [ ]:
Re_a_CL = ϕ_CL_a / sqrt(2*Z_a)
Im_a_CL = q_CL_a * sqrt(Z_a/2)

ϕ_rotating_a =@. ϕ_CL_a * cos(ω_a_CL * t_list_CL) + q_CL_a*Z_a* sin(-ω_a_CL * t_list_CL)
q_rotating_a =@. -ϕ_CL_a/Z_a * sin(-ω_a_CL * t_list_CL) + q_CL_a* cos(ω_a_CL * t_list_CL)

Re_rotating_a_CL = ϕ_rotating_a / sqrt(2*Z_a)
Im_rotating_a_CL = q_rotating_a * sqrt(Z_a/2)

# ϕ_rotating_b =@. ϕ_CL_b * cos(ω_a * t_list) + q_CL_b*Z_a* sin(-ω_a * t_list)
# q_rotating_b =@. -ϕ_CL_b/Z_a * sin(-ω_a * t_list) + q_CL_b* cos(ω_a * t_list);
# p1=plot(Re_a_CL,Im_a_CL)
# p2=plot(Re_rotating_a_CL,Im_rotating_a_CL)

# layout=(1,2)

# plot(p1,p2,layout=layout,
# xlabel=L"\Re(a)", ylabel=L"\Im(a)",
# title=["Resonator a" "Resonator a RF"],
# legend=false,
# size=(layout[2], layout[1]).*size_px_car,
# margin = 10Plots.mm)
#savefig("cat_state_mean_field.pdf")
;

In [ ]:
p1 = plot(t_list_CL*κ, Re_rotating_a_CL,label=L"\Re(a)"* " mean-field")
plot!(t_list*κ, Re_rotating_a,label=L"\Re(a)"* " chain")

p2 = plot(t_list_CL*κ, Im_rotating_a_CL, label=L"\Im(a)"* " mean-field")
plot!(t_list*κ, Im_rotating_a, label=L"\Im(a)"*" chain")

layout=(1,2)

plot(p1,p2,layout=layout,
xlabel=L"\kappa_b t",
legend=true,
size=(layout[2], layout[1]).*size_px_car,
margin = 10Plots.mm)
#savefig("../Figures/Two-photon-plus-one/cat_state_comparison_chain_mean_field.pdf")

In [ ]:
p1 = plot(t_list_CL*κ, Re_rotating_a_CL,label=L"\Re(a)"* " mean-field")
plot!(t_list*κ, Re_rotating_a,label=L"\Re(a)"* " chain")

p2 = plot(t_list_CL*κ, Im_rotating_a_CL, label=L"\Im(a)"* " mean-field")
plot!(t_list*κ, Im_rotating_a, label=L"\Im(a)"*" chain")

layout=(1,2)

plot(p1,p2,layout=layout,
xlabel=L"\kappa_b t",
legend=true,
size=(layout[2], layout[1]).*size_px_car,
margin = 10Plots.mm)
#savefig("../Figures/Two-photon/cat_state_comparison_chain_mean_field.pdf")